# M2 — Consensus Selection (Manuscript, Issue 3C)

**Problem.** The validation set used to pick each run's "best" expression is small (n=1,117,
189 deaths), so it can rank the *class* of GP expressions reliably (30-run mean 0.734 ≈ canonical
0.738) but not individual candidates within that class — hence the weak validation-test
correlation (r = −0.07) reported in the thesis.

**C1 — Consensus selection.** For each of the 30 marathon runs, take the expression at
complexity=24 (all 30 runs have a complexity-24 candidate on their Pareto front, confirmed below).
Score all 30 candidates two ways:
  1. **Validation AUROC/ECE** — already computed and stored in each run's `*_pareto.csv` (from
     the original marathon; no recomputation needed).
  2. **Per-hospital training-set AUROC/ECE** — the *same fixed, already-evolved* expression
     (no retraining) evaluated separately within each of the 65 training-set hospitals. This is
     an independent signal from the small validation set: much larger combined sample (n=7,814),
     broken out by hospital as a robustness check. (Note: this is evaluation-only stratification
     of a fixed model, not a true train/test leave-one-hospital-out procedure — no new GP search
     is run, consistent with the "no new GP runs" constraint for this analysis.)

If the two criteria agree on which run/expression ranks best, selection is validated. If they
disagree, that itself is informative: it means expression selection is genuinely noisy at the
individual-candidate level, and the 30-run *distribution* (C2) is the right primary result.

**C2 — Distribution-first reporting.** Restate the primary GP result as the 30-run mean ± sd
(already computed in `NB11_30run_reproducibility.csv` / used in Issue 3A), with the canonical
seed-14 expression presented explicitly as representative-for-interpretation, not "the winner."

**Does not modify or re-execute any NB01–NB15 thesis notebook or trigger any new GP search.**

In [1]:
import pickle
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.metrics import roc_auc_score
import sys

PROJECT = Path(r"C:\ML PROJECT\sepsis-gp")
sys.path.insert(0, str(PROJECT))
from src.metrics import compute_ece

RUNS_DIR = PROJECT / "results" / "v2_bce" / "gp_runs"
TABLES_SRC = PROJECT / "results" / "v2_bce" / "tables"
OUT_DIR = PROJECT / "results" / "manuscript" / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_RUNS = 30
TARGET_COMPLEXITY = 24
CANONICAL_SEED = 14

with open(PROJECT / "data" / "processed" / "feature_config.json") as f:
    cfg = json.load(f)
GP_TERMINALS = cfg["GP_TERMINALS"] if "GP_TERMINALS" in cfg else None
print("GP_TERMINALS from config:", GP_TERMINALS)

GP_TERMINALS from config: ['age_numeric', 'lactate_max', 'temperature', 'bun', 'heartrate', 'map_mean', 'platelets_min', 'respiratoryrate', 'vent', 'bilirubin', 'pf_ratio', 'gcs_total', 'hematocrit', 'sodium', 'glucose', 'wbc', 'potassium_max', 'ph', 'meanbp', 'creatinine', 'intubated', 'pf_ratio_miss', 'dialysis', 'vasopressor_24h', 'lactate_max_miss', 'gender_male']


In [2]:
# Fall back to the 26-terminal set from gp_terminal_set.csv if feature_config.json
# doesn't expose GP_TERMINALS directly under that key.
if not GP_TERMINALS:
    term_df = pd.read_csv(PROJECT / "data" / "processed" / "gp_terminal_set.csv")
    GP_TERMINALS = term_df["feature"].tolist()
print(f"{len(GP_TERMINALS)} GP terminals: {GP_TERMINALS}")

26 GP terminals: ['age_numeric', 'lactate_max', 'temperature', 'bun', 'heartrate', 'map_mean', 'platelets_min', 'respiratoryrate', 'vent', 'bilirubin', 'pf_ratio', 'gcs_total', 'hematocrit', 'sodium', 'glucose', 'wbc', 'potassium_max', 'ph', 'meanbp', 'creatinine', 'intubated', 'pf_ratio_miss', 'dialysis', 'vasopressor_24h', 'lactate_max_miss', 'gender_male']


In [3]:
split = pd.read_csv(PROJECT / "data" / "processed" / "split_random.csv")
feat = pd.read_parquet(PROJECT / "data" / "processed" / "features_curated.parquet")
# features_curated.parquet already has hospitalid; only merge in the split label.
feat = feat.merge(split[["patientunitstayid", "split"]], on="patientunitstayid")

train_df = feat[feat["split"] == "train"].reset_index(drop=True)
val_df = feat[feat["split"] == "val"].reset_index(drop=True)

X_train = train_df[GP_TERMINALS].copy()
y_train = train_df["hospital_mortality"].to_numpy(dtype=np.float64)
hosp_train = train_df["hospitalid"].to_numpy()

print(f"Train: {len(X_train):,} patients across {train_df['hospitalid'].nunique()} hospitals")
print(f"Val (reference, not reloaded — pareto.csv already has val metrics): {len(val_df):,} patients")

Train: 7,814 patients across 65 hospitals
Val (reference, not reloaded — pareto.csv already has val metrics): 1,117 patients


## Step 1 — Pull each run's complexity-24 validation metrics from its existing pareto.csv

In [4]:
records = []
for seed in range(N_RUNS):
    pareto = pd.read_csv(RUNS_DIR / f"run_{seed:02d}_pareto.csv")
    exact = pareto[pareto["complexity"] == TARGET_COMPLEXITY]
    if len(exact) > 0:
        row = exact.iloc[0]
        complexity_used = TARGET_COMPLEXITY
        exact_match = True
    else:
        pareto["dist"] = (pareto["complexity"] - TARGET_COMPLEXITY).abs()
        row = pareto.sort_values("dist").iloc[0]
        complexity_used = int(row["complexity"])
        exact_match = False
    records.append({
        "run": seed,
        "complexity_used": complexity_used,
        "exact_c24_match": exact_match,
        "val_auroc": row["auroc"],
        "val_ece": row["ece_10bin"],
        "equation": row["equation"],
    })

val_summary = pd.DataFrame(records)
print(f"Runs with exact complexity=24 match: {val_summary['exact_c24_match'].sum()} / {N_RUNS}")
val_summary[["run", "complexity_used", "exact_c24_match", "val_auroc", "val_ece"]]

Runs with exact complexity=24 match: 25 / 30


,run,complexity_used,exact_c24_match,val_auroc,val_ece
0,0,24,True,0.7558,0.0154
1,1,23,False,0.7489,0.0128
2,2,24,True,0.7355,0.0141
3,3,24,True,0.7547,0.0194
4,4,23,False,0.7537,0.0263
5,5,24,True,0.7476,0.0158
6,6,24,True,0.7541,0.0175
7,7,24,True,0.7547,0.0198
8,8,24,True,0.7537,0.0198
9,9,24,True,0.7334,0.0154


## Step 2 — Per-hospital training-set AUROC/ECE for each run's fixed complexity-24 expression

No retraining: load each run's already-fit model, locate the equation row matching
`complexity_used`, generate predictions on the full training set, then break out AUROC/ECE by
hospital. Hospitals with only one outcome class are skipped for AUROC (undefined) but kept for ECE.

In [5]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

loho_records = []
per_hospital_detail = []

for _, rec in val_summary.iterrows():
    seed = int(rec["run"])
    complexity_used = int(rec["complexity_used"])

    with open(RUNS_DIR / f"run_{seed:02d}_model.pkl", "rb") as f:
        model = pickle.load(f)

    match = model.equations_[model.equations_["complexity"] == complexity_used]
    eq_idx = match.index[0]

    raw = model.predict(X_train, index=eq_idx)
    raw = np.where(np.isfinite(raw), raw, 0.0)
    prob = sigmoid(raw)

    hosp_aurocs = []
    n_evaluable = 0
    for h in np.unique(hosp_train):
        mask = hosp_train == h
        yh, ph = y_train[mask], prob[mask]
        if len(np.unique(yh)) < 2:
            continue
        hosp_aurocs.append(roc_auc_score(yh, ph))
        n_evaluable += 1
        per_hospital_detail.append({"run": seed, "hospitalid": h, "n": int(mask.sum()), "auroc": hosp_aurocs[-1]})

    loho_records.append({
        "run": seed,
        "loho_auroc_mean": float(np.mean(hosp_aurocs)),
        "loho_auroc_median": float(np.median(hosp_aurocs)),
        "loho_auroc_sd": float(np.std(hosp_aurocs)),
        "loho_ece_overall": compute_ece(y_train, prob),
        "n_hospitals_evaluable": n_evaluable,
    })

loho_summary = pd.DataFrame(loho_records)
print(f"Hospitals evaluable per run (median): {loho_summary['n_hospitals_evaluable'].median():.0f} / {train_df['hospitalid'].nunique()}")
loho_summary.head()

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


Hospitals evaluable per run (median): 65 / 65


,run,loho_auroc_mean,loho_auroc_median,loho_auroc_sd,loho_ece_overall,n_hospitals_evaluable
0,0,0.727035,0.730357,0.069088,0.011910,65
1,1,0.727817,0.746296,0.071449,0.007112,65
2,2,0.728073,0.744735,0.069025,0.003373,65
3,3,0.730204,0.741514,0.068417,0.009651,65
4,4,0.728855,0.738536,0.068463,0.005781,65


## Step 3 — Combine, rank under both criteria, and check agreement

In [6]:
combined = val_summary.merge(loho_summary, on="run")
combined["rank_val"] = combined["val_auroc"].rank(ascending=False, method="min").astype(int)
combined["rank_loho"] = combined["loho_auroc_mean"].rank(ascending=False, method="min").astype(int)
combined["is_canonical"] = combined["run"] == CANONICAL_SEED

cols = ["run", "is_canonical", "complexity_used", "val_auroc", "rank_val",
        "loho_auroc_mean", "rank_loho", "loho_ece_overall", "n_hospitals_evaluable"]
combined_sorted = combined[cols].sort_values("rank_val")
combined_sorted

,run,is_canonical,complexity_used,val_auroc,rank_val,loho_auroc_mean,rank_loho,loho_ece_overall,n_hospitals_evaluable
14,14,True,24,0.7640,1,0.730058,19,0.008896,65
25,25,False,23,0.7592,2,0.730415,17,0.010445,65
13,13,False,24,0.7585,3,0.737822,2,0.012397,65
12,12,False,24,0.7569,4,0.733287,7,0.003364,65
11,11,False,24,0.7563,5,0.731072,16,0.015070,65
0,0,False,24,0.7558,6,0.727035,27,0.011910,65
21,21,False,24,0.7556,7,0.728635,23,0.009181,65
29,29,False,24,0.7556,7,0.729924,20,0.011428,65
7,7,False,24,0.7547,9,0.732182,11,0.007564,65
3,3,False,24,0.7547,9,0.730204,18,0.009651,65


In [7]:
spearman_r, spearman_p = stats.spearmanr(combined["val_auroc"], combined["loho_auroc_mean"])
kendall_tau, kendall_p = stats.kendalltau(combined["val_auroc"], combined["loho_auroc_mean"])

best_by_val = combined.loc[combined["rank_val"] == 1, "run"].tolist()
best_by_loho = combined.loc[combined["rank_loho"] == 1, "run"].tolist()
canonical_row = combined[combined["run"] == CANONICAL_SEED].iloc[0]

print(f"Spearman rho (val AUROC vs per-hospital-mean AUROC): {spearman_r:.3f}  (p={spearman_p:.4f})")
print(f"Kendall tau                                        : {kendall_tau:.3f}  (p={kendall_p:.4f})")
print(f"\nBest run by validation AUROC   : run {best_by_val}")
print(f"Best run by per-hospital AUROC : run {best_by_loho}")
print(f"\nCanonical (seed={CANONICAL_SEED}) rank under validation criterion  : {int(canonical_row['rank_val'])} / {N_RUNS}")
print(f"Canonical (seed={CANONICAL_SEED}) rank under per-hospital criterion : {int(canonical_row['rank_loho'])} / {N_RUNS}")

Spearman rho (val AUROC vs per-hospital-mean AUROC): 0.271  (p=0.1477)
Kendall tau                                        : 0.157  (p=0.2248)

Best run by validation AUROC   : run [14]
Best run by per-hospital AUROC : run [18]

Canonical (seed=14) rank under validation criterion  : 1 / 30
Canonical (seed=14) rank under per-hospital criterion : 19 / 30


## Step 4 — C2: distribution-first framing (pulling in the already-computed 30-run spread)

In [8]:
nb11_repro = pd.read_csv(TABLES_SRC / "NB11_30run_reproducibility.csv")
gp_repro = nb11_repro[nb11_repro["label"] == "GP"]
print("C2 primary-result framing (i.i.d., 30 independently-partitioned runs, each run's own\n"
      "best-fitness complexity — this is the actual GP procedure, distinct from the fixed-c24\n"
      "robustness check above):\n")
print(f"  30-run mean test AUROC : {gp_repro['auroc'].mean():.4f} +/- {gp_repro['auroc'].std():.4f}")
print(f"  30-run mean test ECE   : {gp_repro['ece_10bin'].mean():.4f} +/- {gp_repro['ece_10bin'].std():.4f}")
print(f"  Canonical (seed=14, complexity=24) test AUROC: 0.738 (reported as representative for\n"
      f"  clinical interpretation, not as \"the selected best\")")

C2 primary-result framing (i.i.d., 30 independently-partitioned runs, each run's own
best-fitness complexity — this is the actual GP procedure, distinct from the fixed-c24
robustness check above):

  30-run mean test AUROC : 0.7339 +/- 0.0221
  30-run mean test ECE   : 0.0162 +/- 0.0041
  Canonical (seed=14, complexity=24) test AUROC: 0.738 (reported as representative for
  clinical interpretation, not as "the selected best")


In [9]:
out_path = OUT_DIR / "M2_consensus_ranking.csv"
combined_sorted.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Saved: C:\ML PROJECT\sepsis-gp\results\manuscript\tables\M2_consensus_ranking.csv


## Findings

Reported after execution below — see printed Spearman/Kendall statistics and the best-run
comparison in Step 3. If rho is weak/non-significant, that confirms individual-expression
selection is genuinely noisy at this validation-set size, and C2's distribution-first framing
(30-run mean ± sd, canonical as representative) is the appropriate primary result — turning
r = −0.07 from a weakness into a documented methodological property of stochastic GP search
under small held-out validation sets, rather than an unexplained anomaly.